# October 14, 2026

## Prep

In [ ]:
import pandas as pd
import numpy as np 
from tabulate import tabulate
from simnibs_parcel_analysis import summarize_treatment_courses

fn_cgi   = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_latest.xlsx'
fn_query = '/projects/p32903/Alex2/ect-results-tools/query_result_1.csv'

def _add_person_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["person_id"] = df["subjid"].str.extract(
        r"^(subj-cat-\d{3})(?:-\d{3})?$", expand=False
    )

    invalid = df["person_id"].isna()
    if invalid.any():
        raise ValueError(
            f"Invalid/missing subjid at rows {df.index[invalid].tolist()}"
        )
    return df

def load_stimulation_data(fn: str) -> pd.DataFrame:
    df = pd.read_excel(fn, sheet_name="stimulus")
    return _add_person_id(df)

def load_course_data(fn: str) -> pd.DataFrame:
    df = pd.read_excel(fn, sheet_name="Sheet1")
    df = df.loc[
        df["in_stim"].eq(True) & df["batch_num"].gt(0)
    ].copy()

    df["cgi_change"] = df["cgi_start"] - df["cgi_end"]
    return _add_person_id(df)

def load_query_results(fn: str) -> pd.DataFrame:
    fs_subjects_dirs = {
        "STU00225089": "/projects/p32903/Alex2/datasets/STU00225089/subjectsdir",
        "STU00225089_synthsr": "/projects/p32903/Alex2/datasets/STU00225089/synthsr/subjectsdir"
    }
    df_query = pd.read_csv(fn)
    df_query["fs_subjects_dir"]         = df_query["dataset_root"].map(fs_subjects_dirs)
    df_query["electrode_configuration"] = df_query["electrode_configuration"].replace("BL", "BT")
    
    if df_query["fs_subjects_dir"].isna().any():
        missing = df_query.loc[df_query["fs_subjects_dir"].isna(), "dataset_root"].unique()
        raise ValueError(f"Missing FreeSurfer SUBJECTS_DIR mapping for: {missing}")
    return df_query

def _compute_ef_sf(df:pd.DataFrame)->pd.DataFrame:
    cols    = ['pulse_width_ms', 'frequency_hz', 'percent_charge']
    df      = df.copy()
    df['ef_sf'] = df['pulse_width_ms'] * df['frequency_hz'] * df['percent_charge']
    return df


In [ ]:
df_query    = load_query_results(fn_query)
df_query    = df_query[df_query['step'].isin(['step_0','step_4'])]
df_query    = df_query[df_query['threshold']==95]
df_query    = df_query[df_query['full_file_path'].str.endswith('tdcs_uq_gpc_magnE_mean.nii.gz', na=False)]

In [ ]:
from pathlib import Path
import nibabel as nib
import numpy as np
import simnibs

# Paths from your existing notebook
ef_path = Path(ef_path)
step_dir = ef_path.parent.parent

mesh_path = step_dir / "tdcs_uq_gpc.hdf5"
out_path = ef_path.with_name("tdcs_uq_gpc_magnE_mean_gm_wm.nii.gz")

# Load original gPC mesh
# mesh = simnibs.read_msh(str(mesh_path))
mesh = simnibs.Msh.read_hdf5(
    "/gpfs/projects/b32903/Alex/results/asf_5/STU00225089/skin_double/subj-cat-012-001/BL/95/20261007_125305/step_0/tdcs_uq_gpc.hdf5",
    "mesh_roi")

# Restrict output to white matter (1) and gray matter (2)
brain = mesh.crop_mesh(tags=[1, 2], elm_type=4)
field = brain.field["magnE_mean"]
values = np.asarray(field.value, dtype=float).reshape(-1)

if values.size != brain.elm.nr or not np.isfinite(values).all() or np.any(values < 0):
    raise ValueError("Invalid magnE_mean field.")

# Verify field availability in each tissue
for tag, name in [(1, "WM"), (2, "GM")]:
    tissue_values = values[np.asarray(brain.elm.tag1) == tag]
    if tissue_values.size == 0 or not np.any(tissue_values > 0):
        raise ValueError(f"Missing positive magnE_mean values in {name}")
    print(f"{name}: {len(tissue_values):,} elements; {100 * np.mean(tissue_values > 0):.2f}% positive")

# Use original NIfTI only as a spatial grid reference
reference = nib.load(str(ef_path))

field.to_nifti(
    reference.shape[:3],
    reference.affine,
    fn=str(out_path),
    method="linear",
    continuous=False,
)

# Validate output grid
exported = nib.load(str(out_path))
if exported.shape != reference.shape or not np.allclose(exported.affine, reference.affine, atol=1e-4):
    raise ValueError("Output grid differs from the reference.")

print(f"Saved: {out_path}")

In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path
from nibabel.processing import resample_from_to

# Paths
ef_path = Path("/gpfs/projects/b32903/Alex/results/asf_5/STU00225089/skin_double/subj-cat-012-001/BL/95/20261007_125305/step_0/subject_volumes/tdcs_uq_gpc_magnE_mean.nii.gz")
new_path = ef_path.with_name("tdcs_uq_gpc_magnE_mean_gm_wm.nii.gz")
seg_path = Path("/projects/p32903/Alex2/datasets/STU00225089/subjectsdir/subj-cat-012-001/mri/aparc+aseg.mgz")

# Load and orient E-field volumes consistently
old_ef = nib.as_closest_canonical(nib.load(str(ef_path)))
new_ef = nib.as_closest_canonical(nib.load(str(new_path)))
seg = nib.load(str(seg_path))

# Verify matching E-field grids
if old_ef.shape != new_ef.shape or not np.allclose(old_ef.affine, new_ef.affine, atol=1e-4):
    raise ValueError("Original and new E-field volumes have different spatial grids.")

old_data = np.asarray(old_ef.dataobj, dtype=float)
new_data = np.asarray(new_ef.dataobj, dtype=float)

for name, data in [("Original", old_data), ("GM+WM", new_data)]:
    if not np.isfinite(data).all() or np.any(data < 0):
        raise ValueError(f"{name} E-field contains invalid values.")

# Resample FreeSurfer segmentation onto E-field grid
seg_resampled = resample_from_to(seg, (new_ef.shape, new_ef.affine), order=0)
labels = np.asarray(seg_resampled.dataobj, dtype=np.int32)

# FreeSurfer subcortical ROIs
roi_ids = {
    "Left-Thalamus": 10, "Right-Thalamus": 49,
    "Left-Caudate": 11, "Right-Caudate": 50,
    "Left-Putamen": 12, "Right-Putamen": 51,
    "Left-Pallidum": 13, "Right-Pallidum": 52,
    "Left-Hippocampus": 17, "Right-Hippocampus": 53,
    "Left-Amygdala": 18, "Right-Amygdala": 54,
    "Left-Accumbens": 26, "Right-Accumbens": 58,
    "Left-VentralDC": 28, "Right-VentralDC": 60,
}

# Calculate coverage per ROI
rows = []

for name, label_id in roi_ids.items():
    mask = labels == label_id
    n_voxels = int(mask.sum())

    if n_voxels == 0:
        raise ValueError(f"Missing ROI after resampling: {name}")

    old_coverage = 100 * np.mean(old_data[mask] > 0)
    new_coverage = 100 * np.mean(new_data[mask] > 0)

    rows.append({
        "roi": name,
        "n_voxels": n_voxels,
        "old_coverage_pct": old_coverage,
        "new_coverage_pct": new_coverage,
        "change_pp": new_coverage - old_coverage,
    })

df_coverage = pd.DataFrame(rows)
display(df_coverage.sort_values("new_coverage_pct").round(2))


import matplotlib.pyplot as plt

df_plot = df_coverage.sort_values("new_coverage_pct")

fig, ax = plt.subplots(figsize=(9, 6))

y = np.arange(len(df_plot))
ax.barh(y - 0.2, df_plot["old_coverage_pct"], height=0.4, label="Original")
ax.barh(y + 0.2, df_plot["new_coverage_pct"], height=0.4, label="GM + WM")

ax.set_yticks(y, df_plot["roi"])
ax.set_xlim(0, 105)
ax.set_xlabel("ROI coverage (%)")
ax.set_title("Subcortical E-field Coverage: Original vs GM+WM")
ax.legend()
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Compare E-field magnitudes at voxels covered by both exports
shared = (old_data > 0) & (new_data > 0) & (labels > 0)

old_values = old_data[shared]
new_values = new_data[shared]

abs_diff = np.abs(new_values - old_values)
relative_diff = 100 * abs_diff / old_values

print(f"Shared voxels: {shared.sum():,}")
print(f"Median absolute difference: {np.median(abs_diff):.4f} V/m")
print(f"P95 absolute difference: {np.percentile(abs_diff, 95):.4f} V/m")
print(f"Median relative difference: {np.median(relative_diff):.2f}%")
print(f"P95 relative difference: {np.percentile(relative_diff, 95):.2f}%")

In [ ]:
abs_diff = np.abs(new_values - old_values)

print(f"Maximum absolute difference: {abs_diff.max():.12g} V/m")
print(f"Mean absolute difference: {abs_diff.mean():.12g} V/m")
print(f"Exactly identical voxels: {100 * np.mean(new_values == old_values):.6f}%")
print(f"Voxels differing > 1e-6 V/m: {np.sum(abs_diff > 1e-6):,}")
print(f"Voxels differing > 1e-4 V/m: {np.sum(abs_diff > 1e-4):,}")

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib

shared = (old_data > 0) & (new_data > 0) & (labels > 0)
diff = np.abs(new_data - old_data)

# Identify differences above numerical tolerance
outlier_mask = shared & (diff > 1e-4)

ijk = np.argwhere(outlier_mask)
xyz = nib.affines.apply_affine(new_ef.affine, ijk)

df_outliers = pd.DataFrame({
    "i": ijk[:, 0],
    "j": ijk[:, 1],
    "k": ijk[:, 2],
    "x_mm": xyz[:, 0],
    "y_mm": xyz[:, 1],
    "z_mm": xyz[:, 2],
    "label_id": labels[outlier_mask],
    "old_E": old_data[outlier_mask],
    "new_E": new_data[outlier_mask],
    "abs_diff": diff[outlier_mask],
})

label_names = {v: k for k, v in roi_ids.items()}
df_outliers["roi"] = df_outliers["label_id"].map(label_names).fillna("Other atlas region")

display(df_outliers.sort_values("abs_diff", ascending=False))

In [ ]:
rows = []

for name, label_id in roi_ids.items():
    mask = shared & (labels == label_id)

    if not mask.any():
        raise ValueError(f"No shared voxels for {name}")

    old_v = old_data[mask]
    new_v = new_data[mask]

    rows.append({
        "roi": name,
        "n_shared": old_v.size,
        "n_different": int(np.sum(np.abs(new_v - old_v) > 1e-4)),
        "delta_mean_E": new_v.mean() - old_v.mean(),
        "delta_p95_E": np.percentile(new_v, 95) - np.percentile(old_v, 95),
        "max_abs_diff": np.max(np.abs(new_v - old_v)),
    })

df_roi_qc = pd.DataFrame(rows)

display(df_roi_qc[df_roi_qc["n_different"] > 0])

In [ ]:
## MAIN

In [ ]:
df_query = load_query_results(fn_query)
df_query = df_query[df_query['full_file_path'].str.endswith('tdcs_uq_gpc.hdf5')]
df_query = df_query[df_query['step'].isin(['step_0', 'step_4'])].copy()# df_ref = step_0, skin_single, threshold 95
df_query = df_query[df_query['threshold']==95]
df_ref = df_query[df_query['step']=='step_0'].copy()
df_cmp = df_query[df_query['step']=='step_4'].copy()

df_ref = df_ref[df_ref['model_type_dir']=='skin_single']
df_cmp = df_cmp[df_cmp['model_type_dir']=='skin_double']

In [ ]:
from tqdm import tqdm
import simnibs
import gc

def _get_magnE_mean_from_hdf(fn: str) -> float:

    mesh = simnibs.Msh.read_hdf5(fn, "mesh_roi")
    # Restrict to white matter (1) and gray matter (2)
    brain = mesh.crop_mesh(tags=[1, 2], elm_type=4)
    field = brain.field["magnE_mean"]
    values = np.asarray(field.value, dtype=float).reshape(-1)
    del mesh, brain, field
    gc.collect()
    return float(values.mean())


def get_magnE_mean_from_hdf(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['E_global_mean'] = [
        _get_magnE_mean_from_hdf(fn)
        for fn in tqdm(
            df['full_file_path'],
            total=len(df),
            desc="Computing E-field means",
            miniters=1
        )
    ]

    return df


df_ref  = get_magnE_mean_from_hdf(df_ref)
df_ref.to_csv('globalE_ref.csv')
df_cmp = get_magnE_mean_from_hdf(df_cmp)
df_cmp.to_csv('globalE_cmp.csv')

In [33]:
# diff = df_ref['E_global_mean'] / df_cmp['E_global_mean']
df_cmp = pd.read_csv('globalE_cmp.csv')
df_ref = df_ref.sort_values(by="subject_dir")

df_cmp = df_cmp.sort_values(by='subject_dir')
df_cmp = df_cmp.reset_index(drop=True)
df_ref = df_ref.reset_index(drop=True)

ratios = df_ref['E_global_mean'].values / df_cmp['E_global_mean'].values

In [37]:
df_ref

,results_root,asf,dataset_root,model_type_dir,subject_dir,electrode_configuration,threshold,sim_result,step,step_result,full_file_path,fs_subjects_dir,E_global_mean
0,results,asf_5,STU00225089_synthsr,skin_single,subj-cat-002-001,BT,95,20261007_143932,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/sy...,285.510200
1,results,asf_5,STU00225089_synthsr,skin_single,subj-cat-002-002,BT,95,20261007_150146,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/sy...,292.465079
2,results,asf_5,STU00225089_synthsr,skin_single,subj-cat-002-003,BT,95,20261007_151255,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/sy...,296.377586
3,results,asf_5,STU00225089_synthsr,skin_single,subj-cat-005-001,BT,95,20261007_151255,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/sy...,299.097484
4,results,asf_5,STU00225089_synthsr,skin_single,subj-cat-011-001,BT,95,20261007_151255,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/sy...,310.399703
5,results,asf_5,STU00225089_synthsr,skin_single,subj-cat-011-001,RUL,95,20261007_151255,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/sy...,379.227558
6,results,asf_5,STU00225089,skin_single,subj-cat-012-001,BT,95,20261007_102014,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/su...,304.489814
7,results,asf_5,STU00225089,skin_single,subj-cat-019-001,RUL,95,20261007_102015,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/su...,407.140197
8,results,asf_5,STU00225089,skin_single,subj-cat-019-001,BT,95,20261007_102014,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/su...,367.889145
9,results,asf_5,STU00225089_synthsr,skin_single,subj-cat-022-001,BT,95,20261007_151845,step_0,NaN,/gpfs/projects/b32903/Alex/results/asf_5/STU00...,/projects/p32903/Alex2/datasets/STU00225089/sy...,325.018562


In [40]:
print("==============================")
print("Ref --> skin_single/step_0")
print("Cmp --> skin_double/step_4")
print("Shared: adaptive threshold: 95%")
print("=== ratios ref / cmp ===")

print(f'mean: {ratios.mean():0.4f}')
print(f'sd: {ratios.std():0.4f}')
print(f'min/max: {ratios.min():0.4f}/{ratios.max():0.4f}')


Ref --> skin_single/step_0
Cmp --> skin_double/step_4
Shared: adaptive threshold: 95%
=== ratios ref / cmp ===
mean: 1.1010
sd: 0.0452
min/max: 0.9880/1.2074
